# 1. `omnipath-client` tour

`omnipath-client` is a lightweight HTTP client for the new OmniPath API
(`dev.omnipathdb.org`). The pilot build serves 26 resources, many of
them small-molecule / metabolomics related (BindingDB, ChEBI, FooDB,
HMDB, Lipid Maps, Reactome, SIGNOR, STITCH, SwissLipids,
WikiPathways, …).

Status: **pre-alpha**, expect rough edges and breaking changes.
A few resources (PTFI, RaMP-DB, ChEMBL) are catalogued but their data
is not yet loaded into the pilot build — we'll skip those examples.

In [ ]:
from _utils import results_dir
import polars as pl
import omnipath_client as oc

## The resource catalog

`oc.resources()` lists every source the API knows about, along with
the kinds of data each contributes (entities, relations,
annotations).

In [ ]:
catalog = oc.resources()
print(f"{len(catalog)} resources")
for r in catalog:
    cats = ", ".join(r.get("categories") or []) or "—"
    print(f"  {r['resource_id']:<18} {r['resource_name']:<32} [{cats}]")

## Two helpers cover most of what you need

Almost every analysis question is one of two shapes:

- *"Tell me about this thing"* — **`oc.lookup()`** resolves a name or
  accession to entity records and pivots the IDs you ask for into
  named columns.
- *"What's connected to this thing?"* — **`oc.related()`** returns a
  joined relations table where both sides already carry names and
  identifiers.

These wrap the lower-level `resolve()` / `entities()` / `relations()`
primitives — the latter are still exported for paged scans, raw
parquet, and graph export, but you rarely need them directly.

## `oc.lookup()` — resolve and enrich

In [ ]:
oc.lookup(
    ["caffeine", "metformin", "glucose"],
    id_types=["name", "chebi", "hmdb", "kegg", "drugbank"],
)

`id_types` accepts friendly aliases (lowercase, snake_case) that the
client maps to the underlying MI/OM ontology codes. Defaults to
`("name", "chebi", "hmdb", "uniprot", "genesymbol")`. When an entity
has several values for the same id type, the shortest is picked.

In [ ]:
oc.lookup(
    "TP53",
    id_types=["name", "uniprot", "genesymbol", "ensembl", "entrez"],
)

## `oc.related()` — joined relations in one call

Pass a string (auto-resolved), a primary key, or a list of either.
Positional argument matches **either** side of the relation;
`subject=` / `object=` pin a direction.
Filters: `sources`, `predicates`, `relation_categories`,
`participant_types` (with aliases like `'protein'`,
`'small_molecule'`).

### Example 1 — Compounds in a strawberry (FooDB)

In [ ]:
oc.related(
    subject="Strawberry",
    sources=["foodb"],
    id_types=["name", "chebi", "hmdb"],
    limit=10,
).select(
    "subject_name", "predicate", "object_name",
    "object_chebi", "object_hmdb", "sources",
)

### Example 2 — Drug targets for caffeine (BindingDB)

In [ ]:
oc.related(
    "caffeine",
    sources=["bindingdb"],
    id_types=["name", "uniprot", "genesymbol"],
).select(
    "subject_name", "predicate", "object_name",
    "object_uniprot", "object_genesymbol",
).head(10)

### Example 3 — Pathway members from WikiPathways and Reactome

`oc.search_terms()` finds pathway IDs by name; `oc.related()` then
pulls the members in one call. `group_by` reorders the rows so each
pathway's members are contiguous.

In [ ]:
hits = oc.search_terms(["glycolysis"], limit=5)
for h in hits["results"]["glycolysis"]:
    print(f"  {h['ontology_id']:<20} {h['id']:<14} {h['name']}")

In [ ]:
oc.related(
    object=["WP253", "R-HSA-70171"],
    sources=["wikipathways", "reactome"],
    relation_categories=["annotation"],
    id_types=["name", "uniprot", "chebi"],
    group_by="object_name",
).select(
    "subject_name", "subject_entity_type", "subject_uniprot",
    "subject_chebi", "predicate", "object_name", "sources",
).head(15)

### Example 4 — Filter by participant type

Friendly aliases (`'protein'`, `'small_molecule'`, …) save you from
remembering the MI codes.

In [ ]:
oc.related(
    "metformin",
    sources=["signor"],
    participant_types=["protein"],
    id_types=["name", "uniprot"],
).select(
    "subject_name", "predicate", "object_name", "object_uniprot",
)

## Quick tour — entity catalogs (no-relation resources)

Some pilot resources contribute only entities (HMDB, Lipid Maps,
SwissLipids, ChEBI). `oc.lookup()` works there too — pass a query
(or a slice of PKs) and ask for the IDs you need.

In [ ]:
for src in ("hmdb", "lipidmaps", "swisslipids", "chebi"):
    df = oc.entities(sources=[src])
    print(f"  {src:<14} {df.height:>8} entities")

In [ ]:
oc.lookup(
    ["palmitic acid", "cholesterol", "phosphatidylcholine"],
    id_types=["name", "chebi", "hmdb", "lipidmaps", "swisslipids"],
)

## Cache control

Responses are cached on disk by default. Two helpers manage that:

- `oc.cache_clear()` removes every cached entry (incl. the OpenAPI
  spec) — useful when the server has just been re-deployed.
- `with oc.fresh(): …` re-downloads each request the **first** time
  it's seen inside the block, then serves later identical requests
  from the freshly populated cache. Useful when you want fresh data
  for a specific analysis without nuking the whole cache.

In [ ]:
with oc.fresh():
    df = oc.related("caffeine", sources=["bindingdb"], limit=5)
df.height

## Underneath: the primitives

`lookup()` and `related()` are wrappers; the four building blocks are
still there when you need raw tables, paging, or graph export:
`oc.resolve(['name'])`, `oc.entities(sources=[…], entity_pks=[…])`,
`oc.relations(sources=[…], …, as_graph=False)`,
`oc.search_terms([…])`. Plus `oc.entities_slice()` and
`oc.relations_slice()` for paged access.

## Recap

- `oc.resources()` — what's in the catalog.
- `oc.lookup(query, id_types=…)` — resolve to entity record(s) with
  the ID columns already pivoted.
- `oc.related(query, sources=…, id_types=…, …)` — joined relations
  table around a query in one call. Filters: `sources`, `predicates`,
  `relation_categories`, `participant_types`; `subject=`/`object=`
  pin direction; `group_by=` reorders; `limit=` truncates.
- `oc.cache_clear()` and `with oc.fresh(): …` for cache control.

Script 02 uses the utils API for ID translation and orthology
mapping.

# 2. ID translation & orthology — the utils API

`omnipath_client.utils` is a thin wrapper around the
`utils.omnipathdb.org` web service. It serves the same translation /
taxonomy / orthology logic that the legacy R `OmnipathR::translate_ids()`
wraps locally — but as an HTTP service, so you don't need to download
the resource files.

This is the Python parallel of the R `03_id_translation.R` section.

In [ ]:
from _utils import results_dir
from omnipath_client import utils

## Available ID types

`id_types()` returns a list of `{name, label, entity_type, curie_prefix}`
dicts covering proteins, small molecules, genes, etc.

In [ ]:
all_types = utils.id_types()
print(f"{len(all_types)} ID types in total")
[t for t in all_types if t["entity_type"] == "small_molecule"][:8]

## Translate one identifier

In [ ]:
utils.map_name("TP53", "genesymbol", "uniprot")

## Translate many at once

`map_names()` returns the union; `translate_column()` keeps the
row-to-row mapping as a DataFrame.

In [ ]:
ids = ["TP53", "MYC", "CDK2", "BRCA1"]
utils.map_names(ids, "genesymbol", "uniprot")

In [ ]:
df = utils.translation_df("genesymbol", "uniprot", identifiers=ids)
df

## Metabolite ID translation

The same mechanism handles small-molecule IDs.

In [ ]:
utils.map_name("HMDB0000094", "hmdb", "chebi")  # citrate

In [ ]:
metabolites = ["HMDB0000094", "HMDB0000254", "HMDB0000190"]  # citrate, succinate, lactate
utils.translation_df("hmdb", "chebi", identifiers=metabolites)

## Taxonomy resolution

Names, IDs, codes — `resolve_organism` handles them all.

In [ ]:
utils.ensure_ncbi_tax_id("human"), utils.ensure_ncbi_tax_id("Mus musculus"), utils.ensure_ncbi_tax_id(10090)

## Orthology

Translate a list of human gene symbols to mouse.

In [ ]:
from omnipath_client.utils._orthology import orthology_df

orthology_df(
    source=9606,
    target=10090,
    id_type="genesymbol",
    identifiers=["TP53", "MYC", "CDK2"],
)

**Recap.** ID translation, taxonomy resolution, and orthology — all via
a small handful of utility calls. This is the bridge that lets us
combine human-only resources with multi-organism analyses, which we'll
use when fetching COSMOS PKN data in script 03.

# 3. COSMOS PKN — fetched via the client

**COSMOS** is a multi-omics mechanistic-modelling framework that
combines metabolomics, signalling, and transcriptomics. Its Prior
Knowledge Network (PKN) is a multi-layer graph stitched together from:

- **transporters** — TCDB, SLC, GEM, Recon3D, MRCLinksDB
- **receptors** — MRCLinksDB, STITCH
- **allosteric regulation** — BRENDA, STITCH
- **enzyme ↔ metabolite** — Human-GEM, Recon3D
- **PPI** — OmniPath
- **GRN** — CollecTRI

We won't run COSMOS itself today, but we'll inspect the PKN and the
GEM-derived layer because the binary network (reaction → reactant /
product) is generally useful for any metabolic-network reasoning.

In [ ]:
from _utils import results_dir
from omnipath_client import cosmos

## What's available?

In [ ]:
cosmos.categories()

In [ ]:
cosmos.organisms()

In [ ]:
cosmos.resources(organism="human")

## Status of the live build

In [ ]:
cosmos.status()

## Fetch the enzyme ↔ metabolite layer

This is the layer that captures "enzyme E catalyses a reaction
producing/consuming metabolite M". The server delivers it pre-cleaned
(canonical ChEBI / UniProt IDs, metabolic vs transport split).

In [ ]:
em_df = cosmos.get_pkn(
    organism="human",
    categories="enzyme_metabolite",
    format="dataframe",
)
em_df.head()

In [ ]:
print(f"{em_df.height} rows × {em_df.width} columns")
em_df.columns

## A small subnetwork for visualisation

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Take a small slice to keep the plot readable.
import polars as pl

slice_df = em_df.head(80).select(["source", "target", "interaction_type"])
G = nx.from_pandas_edgelist(
    slice_df.to_pandas(),
    source="source",
    target="target",
    edge_attr="interaction_type",
    create_using=nx.DiGraph,
)

fig, ax = plt.subplots(figsize=(9, 7))
pos = nx.spring_layout(G, seed=42, k=0.7)
nx.draw_networkx_nodes(G, pos, node_size=80, node_color="#88aadd", ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.4, arrows=True, arrowsize=8, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=6, ax=ax)
ax.set_title("First 80 enzyme ↔ metabolite edges of the COSMOS PKN")
ax.set_axis_off()
fig.savefig(results_dir("03_cosmos") / "cosmos_em_subnet.png", dpi=150, bbox_inches="tight")
plt.show()

## The other categories at a glance

Each is one HTTP call away; the API is uniform.

In [ ]:
for cat in ["transporters", "receptors", "allosteric"]:
    df = cosmos.get_pkn(organism="human", categories=cat)
    print(f"{cat:<14s}  {df.height:>6d} edges")

**Recap.** Three lines of Python and we have a multi-layer
metabolism-aware network. Script 04 shows what the GEM looks like
*before* COSMOS simplifies it.

# 4. Raw GEM reactions via `omnipath-metabo`

`omnipath-metabo` is the server-side builder behind the COSMOS PKN.
Hitting it directly (rather than the client) lets us see resources in
their original shape — before COSMOS simplifies them into a binary
enzyme ↔ metabolite graph.

We focus on **GEM (Genome-scale Metabolic Model) reactions** — these
carry stoichiometry, compartment information, and reaction reversibility,
which the COSMOS-formatted version largely flattens away.

In [ ]:
from _utils import results_dir
from omnipath_metabo.datasets.cosmos.resources import gem_interactions

## A few raw interactions

`gem_interactions()` is a generator that yields one `Interaction`
named-tuple per edge.

In [ ]:
gem_iter = gem_interactions(gem="Human-GEM", organism=9606)
sample = [next(gem_iter) for _ in range(5)]
for rec in sample:
    print(rec)

Each record carries:

- `source` / `target` — IDs (ChEBI for metabolites, UniProt for proteins)
- `source_type` / `target_type` — `'small_molecule'` / `'protein'`
- `interaction_type` — what the edge represents (e.g. `'catalyses'`,
  `'transports'`)
- `resource` — `'GEM:Human-GEM'` for metabolic reactions,
  `'GEM_transporter:Human-GEM'` for transport reactions
- `locations` — compartment(s) where the reaction occurs
- `mor` — mode-of-regulation (sign, where applicable)
- `attrs` — extra resource-specific fields

## Collect everything into a DataFrame

Be aware: this materialises tens of thousands of edges; takes a few
seconds.

In [ ]:
import pandas as pd

records = list(gem_interactions(gem="Human-GEM", organism=9606))
print(f"Total Human-GEM interactions: {len(records)}")

df = pd.DataFrame(
    {
        "source": [r.source for r in records],
        "target": [r.target for r in records],
        "interaction_type": [r.interaction_type for r in records],
        "resource": [r.resource for r in records],
        "locations": [r.locations for r in records],
        "mor": [r.mor for r in records],
    }
)
df.head()

## Metabolic vs transport split

In [ ]:
df["resource"].value_counts()

## Compartment distribution

GEM reactions are localised to a compartment (cytosol, mitochondrion,
extracellular …). Transport reactions cross compartments.

In [ ]:
from collections import Counter

loc_counter: Counter[str] = Counter()
for locs in df["locations"]:
    if locs:
        for loc in locs:
            loc_counter[loc] += 1

pd.Series(loc_counter).sort_values(ascending=False).head(15)

## Slice: a single metabolite's neighbourhood

Pick citrate (`CHEBI:30769`) and look at every reaction it touches.

In [ ]:
citrate = "CHEBI:30769"
near_citrate = df[(df["source"] == citrate) | (df["target"] == citrate)]
print(f"{len(near_citrate)} reactions involve citrate")
near_citrate.head(10)

In [ ]:
df.to_parquet(results_dir("04_gem") / "human_gem_interactions.parquet")

**Recap.** Same data the client serves under `enzyme_metabolite`, but
with stoichiometry and compartment information intact. Useful when you
need to reason about reaction direction, mass balance, or
compartmentalisation rather than just connectivity.

# 5. (OPTIONAL) BRENDA allosteric regulation

**Skip if running short on time.** This section showcases a single
resource — BRENDA — whose depth and curation quality make it worth a
brief stop.

BRENDA catalogues enzyme function with manual literature curation: each
regulator is annotated with the reference(s) and, often, with the
specific kinetic effect. For metabolomics modelling, this is one of the
few places you can find systematic small-molecule → enzyme allosteric
regulation data.

In [ ]:
from _utils import results_dir
from omnipath_metabo.datasets.cosmos.resources import brenda_regulations

## A few raw regulations

In [ ]:
sample = []
gen = brenda_regulations(organism=9606)
for _ in range(5):
    sample.append(next(gen))

for rec in sample:
    print(rec)

## Collect everything

In [ ]:
import pandas as pd

records = list(brenda_regulations(organism=9606))
print(f"Total human BRENDA allosteric regulations: {len(records)}")

df = pd.DataFrame(
    {
        "regulator": [r.source for r in records],
        "enzyme": [r.target for r in records],
        "interaction_type": [r.interaction_type for r in records],
        "mor": [r.mor for r in records],
        "resource": [r.resource for r in records],
    }
)
df.head()

## Mode of regulation distribution

`mor` (mode-of-regulation) carries the sign: activation, inhibition, or
unknown.

In [ ]:
df["mor"].value_counts(dropna=False)

## Most-regulated enzymes

In [ ]:
df.groupby("enzyme").size().sort_values(ascending=False).head(15)

## Most "active" regulators

In [ ]:
df.groupby("regulator").size().sort_values(ascending=False).head(15)

In [ ]:
df.to_parquet(results_dir("05_brenda") / "brenda_allosteric.parquet")

**Wrap up.**

Today we walked from raw metabolomics measurements through differential
analysis, ID translation, prior-knowledge access, enrichment, and into
multi-layer mechanistic networks. Two takeaways:

1. The R side (OmnipathR + MetaProViz) is now a coherent SE-centric
   pipeline; the API is stable, vignettes are up to date, and a
   Bioconductor release lands shortly.
2. The Python side (omnipath-client + omnipath-metabo) is fresh and
   pre-alpha. Bug reports welcome at
   [github.com/saezlab](https://github.com/saezlab).